In [3]:
import dspy
from typing import Literal

# --- your existing signature ---
class ClassifyLymeAbstract(dspy.Signature):
        """
        Determine the most accurate classification for a Lyme disease–related paper
        when three classifiers provide differing opinions. Use the title, abstract,
        and the reasoning from each classifier to make a final decision.
        """

        abstract: str = dspy.InputField(
            desc="Abstract text of the paper."
        )

        class1: str = dspy.InputField(
            desc="Predicted class from Classifier 1."
        )
        reason1: str = dspy.InputField(
            desc="Justification provided by Classifier 1 for its prediction."
        )

        class2: str = dspy.InputField(
            desc="Predicted class from Classifier 2."
        )
        reason2: str = dspy.InputField(
            desc="Justification provided by Classifier 2 for its prediction."
        )

        class3: str = dspy.InputField(
            desc="Predicted class from Classifier 3."
        )
        reason3: str = dspy.InputField(
            desc="Justification provided by Classifier 3 for its prediction."
        )

        classification: Literal[
                "Supports PTLDS",
                "Supports CLD",
                "Neutral",
                "Unrelated",
                "Animal Study"
        ] = dspy.OutputField(desc="""Classes
        """)
        confidence: Literal["High", "Medium", "Low"] = dspy.OutputField(
                desc="Confidence level in the classification."
        )
        
        reason: str = dspy.OutputField(
                desc="2–3 sentence justification for the classification, indicating explicit or implicit stance."
        )

In [4]:
# load json from filepath
import json
import os
# student_llm_string = "openai/gpt-4o-mini"
# student_llm_string = "openai/gpt-oss-20b"
student_llm_string = 'openrouter/x-ai/grok-4-fast'
API_KEY = os.getenv("openrouter_api_key")
student_lm = dspy.LM(
    model=student_llm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # (plus any model_params like temperature, max_tokens, etc)
    # temperature=1.0, top_p=1.0, seed=42
    temperature=1.0, max_tokens = 20000,
)
dspy.configure(lm=student_lm)

classifier_decision = dspy.ChainOfThought(ClassifyLymeAbstract)

# path1 = '../results/results_stuedent_openai_gpt-5-minireflactoropenai_gpt-4o-mini_original_gepa_auto_medium.json'
# path2 = '../results/results_stuedent_openai_gpt-oss-20breflactoropenai_gpt-oss-20b_gepa_auto_medium.json'
# path3 = '../results/results_stuedent_openai_gpt-4o-minireflactoropenai_gpt-5-mini_original_gepa_auto_medium.json'
# path1 = '../results/results_stuedent_openai_gpt-oss-20breflactoropenai_gpt-oss-20b_no_training.json'
# path2 = '../results/results_stuedent_openai_gpt-oss-20breflactoropenai_gpt-oss-20b_original_gepa_auto_medium.json'
# path3 = '../results/results_stuedent_openai_gpt-oss-20breflactoropenai_gpt-oss-20b_original_no_training.json'
# path3 = '../results/original_promp_openai_gpt-4o-mini.json'


# path1 =    "../results/classification/stuedent_openai_gpt-4o-minireflactoropenai_gpt-5-mini_original_gepa_auto_medium.json"
# path2 =    "../results/classification/stuedent_openai_gpt-5-minireflactoropenai_gpt-4o-mini_original_gepa_auto_medium.json"
# path3 =    "../results/classification/stuedent_openai_gpt-oss-20breflactoropenai_gpt-oss-20b_gepa_auto_medium.json"


path1 =    "../results/classification/stuedent_openrouter_google_gemini-2.0-flash-001reflactoropenrouter_google_gemini-2.0-flash-001_original_gepa_auto_medium.json"
path2 =    "../results/classification/stuedent_openai_gpt-5-minireflactoropenai_gpt-4o-mini_original_gepa_auto_medium.json"
path3 =    "../results/classification/stuedent_openrouter_x-ai_grok-4-fastreflactoropenrouter_x-ai_grok-4-fast_original_gepa_auto_medium.json"

with open(path1, 'r') as f:
    data1 = json.load(f)
with open(path2, 'r') as f:
    data2 = json.load(f)
with open(path3, 'r') as f:
    data3 = json.load(f)
data1 = data1['results']
data2 = data2['results']
data3 = data3['results']
final_data = []
for i in range(len(data1)):
    ensembled_decison_failes = False
    final_class_reseaoning = "N/A"
    class1 = data1[i]['predicted_decision']
    class2 = data2[i]['predicted_decision']
    class3 = data3[i]['predicted_decision']
    # choose class that was selected by at least 2 models
    if class1 == class2 or class1 == class3:
        final_class = class1
    elif class2 == class3:
        final_class = class2
    else:
        ensembled_decison_failes = True
        print("All classes different:", class1, class2, class3)
        prediction = classifier_decision(
            abstract = data1[i]["abstract"],
            class1 = data1[i]["predicted_decision"],
            reason1 = data1[i]["predicted_reasoning"],
            class2 = data2[i]["predicted_decision"],
            reason2 = data2[i]["predicted_reasoning"],
            class3 = data3[i]["predicted_decision"],
            reason3 = data3[i]["predicted_reasoning"],
        )
        # final_class = class1
        final_class = prediction.classification
        final_class_reseaoning = prediction.reason
        print(final_class)
        print(data1[i]['ground_truth_decision'])
    predicitoi_correct = (final_class == data1[i]['ground_truth_decision'])
    # print("prediciton was correct:", predicitoi_correct)
    
    # print(f"Predictions: {class1}, {class2}, {class3}")
    # print(f"Ground truth: {data1[i]['ground_truth_decision']}")
    final_data.append({
        "abstract": data1[i]['abstract'],
        'predicted_decision': final_class,
        'predicition_correct': predicitoi_correct,
        'ensembled_decison_failes': ensembled_decison_failes,
        'ground_truth_decision': data1[i]['ground_truth_decision'],
        'final_reasoning': final_class_reseaoning,
    })
# calcuulate accuracy
accuracy = sum([1 for item in final_data if item['predicition_correct']]) / len(final_data)
print(f"Ensembled accuracy: {accuracy}")
print(len(final_data))
with open('../results/classification/ensembled_classification_results.json', 'w') as f:
    json.dump({'results': final_data}, f, indent=4)



All classes different: Supports CLD Animal Study Unrelated
Animal Study
Unrelated
All classes different: Neutral Unrelated Supports PTLDS
Unrelated
Neutral
All classes different: Neutral Animal Study Supports CLD
Supports CLD
Unrelated
All classes different: Neutral Unrelated Supports CLD
Neutral
Unrelated
All classes different: Supports CLD Supports PTLDS Unrelated
Unrelated
Supports PTLDS
All classes different: Supports CLD Neutral Unrelated
Unrelated
Supports PTLDS
All classes different: Neutral Unrelated Supports CLD
Neutral
Unrelated
All classes different: Neutral Supports PTLDS Unrelated
Supports PTLDS
Unrelated
All classes different: Neutral Unrelated Supports CLD
Unrelated
Neutral
All classes different: Neutral Supports PTLDS Unrelated
Supports PTLDS
Unrelated
All classes different: Supports CLD Neutral Unrelated
Neutral
Neutral
All classes different: Supports CLD Unrelated Neutral
Neutral
Unrelated
All classes different: Supports PTLDS Unrelated Neutral
Neutral
Unrelated
All c